# 04 - RAG Avancado
> Chunking, HyDE e reranking

**Modulo:** `06_rag` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
def chunk(texto,tam=400,overlap=50):
    chunks,i=[],0
    while i<len(texto):
        c=texto[i:i+tam]
        if i+tam<len(texto):
            p=max(c.rfind('. '),c.rfind('\n'))
            if p>tam//2: c=c[:p+1]
        chunks.append(c.strip()); i+=len(c)-overlap
    return [c for c in chunks if c]

t='Python e otimo para ciencia de dados. '*10+'Docker e essencial em producao. '*10
cs=chunk(t)
print(f'{len(t)} chars -> {len(cs)} chunks')
for i,c in enumerate(cs[:2]): print(f'Chunk {i}: "{c[:60]}..."')

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

model=SentenceTransformer('all-MiniLM-L6-v2')
col=chromadb.Client().get_or_create_collection('hyde')
docs=['Python criado em 1991.','FastAPI e rapido.','Docker containeriza apps.']
col.upsert(ids=[f'd{i}'for i in range(len(docs))],embeddings=model.encode(docs).tolist(),documents=docs)

def hyde_busca(q,n=2):
    hip=ask(f'Responda brevemente como documento tecnico: {q}',model=HAIKU)
    res=col.query(query_embeddings=model.encode([hip]).tolist(),n_results=n)
    return res['documents'][0], hip

docs_enc,hip=hyde_busca('Quando Python foi criado?')
print(f'Hipotetica: {hip[:80]}')
print(f'Resultado: {docs_enc[0]}')

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
